# Experiment 3 — Statistics & Paper-Ready Outputs

Takes the result zips from notebooks 1 & 2 and produces:
1. **Bootstrap 95% CIs** on fertility spreads (per tokenizer) and retrieval means (per encoder, per language).
2. **Formal correlations** of both inequity axes (tokenizer fertility, retrieval accuracy) with **Joshi resource-tier** and **script type** (Spearman + p-values; Kruskal-Wallis across scripts).
3. **Paper-ready outputs**: LaTeX tables (booktabs) and a drafted Results section tying Exp 1 (tokenizer fairness, Llama-3 as live negative example) and Exp 2 (low-resource retrieval benchmark) via the **two-axes-of-inequity** framing.

**Runtime:** CPU, < 2 min. No model downloads.

## 0. Install + upload the two result zips

In [ ]:
!pip -q install pandas numpy scipy 2>/dev/null
import os, io, json, zipfile, glob
import numpy as np, pandas as pd
from scipy import stats
rng=np.random.default_rng(0)
print('ready')

In [ ]:
from google.colab import files
WORK='/content/res'; os.makedirs(WORK, exist_ok=True)
print('Upload BOTH: quran_fertility_results.zip and quran_retrieval_results.zip')
up=files.upload()
for fn in up:
    if fn.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(up[fn])) as z: z.extractall(WORK)
        print('extracted', fn)
def find(name):
    hits=glob.glob(os.path.join(WORK,'**',name), recursive=True)
    return hits[0] if hits else None
prod=pd.read_csv(find('fertility_production.csv'))
ctrl=pd.read_csv(find('fertility_controlled.csv'))
rpair=pd.read_csv(find('retrieval_results.csv'))
rlang=pd.read_csv(find('retrieval_per_language.csv'))
print('fertility langs:', len(prod), '| retrieval pairs:', len(rpair))

## 1. Language metadata — Joshi resource-tier + script

**EDIT THIS CELL.** Joshi classes are my best-effort reading of Joshi et al. (2020) "The State and Fate of Linguistic Diversity" (class 0 = least-resourced ... 5 = English). **Verify each value against their Table 1 before using in the paper.** Script labels are objective but check the two flagged cases (Kurdish, Azeri) for your specific translations.

In [ ]:
# joshi: 0..5 (5=most resourced). VERIFY against Joshi et al. 2020 Table 1.
JOSHI={
 'english':5,'arabic_source':5,'chinese':5,'french':5,'german':5,'japanese':5,'spanish':5,
 'portuguese':4,'dutch':4,'italian':4,'persian':4,'turkish':4,'vietnamese':4,'indonesian':3,
 'urdu':3,'tamil':3,'romanian':3,'hindi':4,'serbian':3,'croatian':3,'bosnian-rwwad':3,
 'bosnian-mihanovich':3,'macedonian':1,'bulgarian':3,'azeri':1,'albanian':1,'lithuanian':3,
 'malayalam':3,'kannada':3,'gujarati':3,'assamese':1,'khmer':1,'kurdish':1,'kyrgyz':1,
 'tajik':1,'uzbek':1,'uyghur':1,'pashto':1,'hausa':2,'yoruba':2,'oromo':1,'lingala':1,
 'moore':0,'asante':1,'tagalog':3,
}
# script: Latin / Cyrillic / Arabic / Brahmic / Khmer / CJK
SCRIPT={
 'english':'Latin','french':'Latin','german':'Latin','spanish':'Latin','portuguese':'Latin',
 'dutch':'Latin','indonesian':'Latin','vietnamese':'Latin','turkish':'Latin','romanian':'Latin',
 'albanian':'Latin','croatian':'Latin','bosnian-rwwad':'Latin','bosnian-mihanovich':'Latin',
 'lithuanian':'Latin','hausa':'Latin','yoruba':'Latin','oromo':'Latin','lingala':'Latin',
 'moore':'Latin','asante':'Latin','tagalog':'Latin','uzbek':'Latin','azeri':'Latin',
 'macedonian':'Cyrillic','serbian':'Cyrillic','kyrgyz':'Cyrillic','tajik':'Cyrillic',
 'arabic_source':'Arabic','persian':'Arabic','urdu':'Arabic','pashto':'Arabic','uyghur':'Arabic',
 'kurdish':'Arabic',
 'assamese':'Brahmic','gujarati':'Brahmic','kannada':'Brahmic','malayalam':'Brahmic','tamil':'Brahmic',
 'khmer':'Khmer','chinese':'CJK','japanese':'CJK',
}
meta=pd.DataFrame({'language':sorted(set(prod.language))})
meta['joshi']=meta.language.map(JOSHI)
meta['script']=meta.language.map(SCRIPT)
missing=meta[meta.joshi.isna() | meta.script.isna()]
if len(missing): print('WARNING: fill metadata for:', missing.language.tolist())
else: print('metadata complete for all', len(meta), 'languages')
meta

## 2. Bootstrap CIs

**Retrieval (strong):** resample the per-pair top-1 values with replacement → 95% CI on each encoder's mean.
**Fertility (over languages):** the CSV stores one mean tokens/verse per language, so we bootstrap *over languages* to get a CI on each tokenizer's cross-language mean and spread. (A per-verse CI would need the raw counts — see the optional cell at the end.)

In [ ]:
B=10000
def ci(arr, fn=np.mean, b=B):
    arr=np.asarray(arr,dtype=float); arr=arr[~np.isnan(arr)]
    if len(arr)==0: return (np.nan,np.nan,np.nan)
    idx=rng.integers(0,len(arr),size=(b,len(arr)))
    boot=fn(arr[idx],axis=1)
    return (fn(arr), np.percentile(boot,2.5), np.percentile(boot,97.5))

# --- retrieval: per-encoder mean top1 with CI (bootstrap over pairs) ---
rows=[]
for m,g in rpair.groupby('model'):
    mean,lo,hi=ci(g['top1'].values)
    mm,ml,mh=ci(g['mrr'].values)
    rows.append({'encoder':m.split('/')[-1],'top1':mean,'top1_lo':lo,'top1_hi':hi,'mrr':mm,'mrr_lo':ml,'mrr_hi':mh})
enc_ci=pd.DataFrame(rows).sort_values('top1',ascending=False)
enc_ci.to_csv('retrieval_encoder_ci.csv',index=False)
print('=== Encoder mean top-1 with 95% CI (bootstrap over pairs) ===')
print(enc_ci.to_string(index=False,float_format=lambda x:'%.3f'%x))

In [ ]:
# --- per-language retrieval mean top1 with CI (bootstrap over pairs touching that language) ---
key = 'src' if 'src' in rpair.columns else rpair.columns[1]
rows=[]
for lang,g in rpair.groupby(key):
    mean,lo,hi=ci(g['top1'].values)
    rows.append({'language':lang,'mean_top1':mean,'lo':lo,'hi':hi,'n_pairs':len(g)})
lang_ci=pd.DataFrame(rows).sort_values('mean_top1')
lang_ci.to_csv('retrieval_language_ci.csv',index=False)
print('Worst 10 languages (mean top-1, 95% CI):')
print(lang_ci.head(10).to_string(index=False,float_format=lambda x:'%.3f'%x))

In [ ]:
# --- fertility: per-tokenizer cross-language mean (proper CI) + spread point estimate ---
# Bootstrapping max/min (spread) over languages is biased: resampling can only shrink the
# range, so the spread CI is degenerate. We therefore report a bootstrap CI on the MEAN
# tokens/verse (what was requested), give spread as a point estimate, and add a p90/p10
# percentile-ratio as a robust inequality measure that DOES bootstrap correctly.
tok=[c for c in prod.columns if c.endswith('_tok_per_verse')]
sub=prod[prod.language!='arabic_source']
def p9010(a, axis=None):
    return np.percentile(a,90,axis=axis)/np.percentile(a,10,axis=axis)
rows=[]
for c in tok:
    vals=sub[c].dropna().values
    mean,mlo,mhi=ci(vals)                      # valid bootstrap CI on the mean
    r,rlo,rhi=ci(vals, fn=p9010)               # valid bootstrap CI on p90/p10 ratio
    rows.append({'tokenizer':c.replace('_tok_per_verse',''),
                 'mean_tpv':mean,'mean_lo':mlo,'mean_hi':mhi,
                 'spread_maxmin':vals.max()/vals.min(),         # point estimate only
                 'p90_p10':r,'p90_p10_lo':rlo,'p90_p10_hi':rhi})
fert_ci=pd.DataFrame(rows).sort_values('p90_p10')
fert_ci.to_csv('fertility_tokenizer_ci.csv',index=False)
print('=== Tokenizer fertility: mean tokens/verse (95% CI) + inequality measures ===')
print('spread_maxmin = point estimate (CI omitted: biased under resampling)')
print('p90_p10 = robust inequality ratio with valid 95% bootstrap CI')
print(fert_ci.to_string(index=False,float_format=lambda x:'%.2f'%x))

## 3. Formal correlation with Joshi tier and script type

Build a per-language table (GPT-4o fertility, mean retrieval) joined to metadata, then:
- **Spearman** of each axis vs Joshi tier (ordinal) with p-values.
- **Kruskal-Wallis** of each axis across script groups (categorical), with group medians.

In [ ]:
# per-language retrieval mean (avg over encoders) from rlang
rl = rlang.groupby('language')['mean_top1'].mean().rename('retrieval')
fcol='gpt4o_o200k_tok_per_verse' if 'gpt4o_o200k_tok_per_verse' in prod.columns else [c for c in prod.columns if c.endswith('_tok_per_verse')][0]
df=prod[['language',fcol]].rename(columns={fcol:'fertility'}).merge(meta,on='language').merge(rl,on='language',how='left')
df=df[df.language!='arabic_source']
print('languages in joint table:', len(df))

print('\n=== Spearman vs Joshi tier (higher tier = more resourced) ===')
for ax in ['fertility','retrieval']:
    d=df[['joshi',ax]].dropna()
    rho,p=stats.spearmanr(d['joshi'], d[ax])
    print('  %-10s rho=%+.3f  p=%.2g  (N=%d)'%(ax,rho,p,len(d)))

print('\n=== Fertility vs Retrieval (the two axes) ===')
d=df[['fertility','retrieval']].dropna()
rho,p=stats.spearmanr(d.fertility,d.retrieval)
print('  Spearman rho=%+.3f  p=%.2g  N=%d'%(rho,p,len(d)))

In [ ]:
print('=== Kruskal-Wallis across script groups ===')
for ax in ['fertility','retrieval']:
    groups=[g[ax].dropna().values for _,g in df.groupby('script') if g[ax].notna().sum()>0]
    H,p=stats.kruskal(*groups)
    print('  %-10s H=%.2f  p=%.2g'%(ax,H,p))
print('\n=== Median by script ===')
med=df.groupby('script')[['fertility','retrieval']].median().sort_values('fertility')
print(med.to_string(float_format=lambda x:'%.3f'%x))
df.to_csv('joint_language_table.csv',index=False)
print('\nsaved joint_language_table.csv')

## 4. Paper-ready LaTeX tables (booktabs)

In [ ]:
def latex_table(dfx, caption, label, floatfmt='%.2f'):
    cols=list(dfx.columns)
    out=['\\begin{table}[t]','\\centering','\\small',
         '\\begin{tabular}{l'+'r'*(len(cols)-1)+'}','\\toprule',
         ' & '.join(c.replace('_','\\_') for c in cols)+' \\\\','\\midrule']
    for _,r in dfx.iterrows():
        cells_=[str(r[cols[0]])]+[ (floatfmt%r[c] if isinstance(r[c],(int,float,np.floating)) and pd.notna(r[c]) else str(r[c])) for c in cols[1:]]
        out.append(' & '.join(cells_)+' \\\\')
    out+=['\\bottomrule','\\end{tabular}','\\caption{'+caption+'}','\\label{'+label+'}','\\end{table}']
    return '\n'.join(out)

t1=fert_ci[['tokenizer','mean_tpv','mean_lo','mean_hi','spread_maxmin','p90_p10','p90_p10_lo','p90_p10_hi']].copy()
tex1=latex_table(t1,'Tokenizer fertility on identical content: cross-language mean tokens/verse and mean tokens/verse with 95\\% bootstrap CI, max/min spread (point), and p90/p10 inequality ratio with 95\\% CI. Lower = fairer.','tab:fertility')
t2=enc_ci[['encoder','top1','top1_lo','top1_hi','mrr']].copy()
tex2=latex_table(t2,'Cross-lingual verse retrieval: mean top-1 and MRR per encoder with 95\\% bootstrap CIs.','tab:retrieval')
open('table_fertility.tex','w').write(tex1)
open('table_retrieval.tex','w').write(tex2)
print(tex1); print(); print(tex2)

## 5. Drafted Results section (folds both experiments + two-axes framing)

In [ ]:
def f(lbl,col,d=df):
    r=d[d.language==lbl]; return r[col].iloc[0] if len(r) else float('nan')
gpt4o=fert_ci[fert_ci.tokenizer=='gpt4o_o200k']
llama=fert_ci[fert_ci.tokenizer=='llama3']
cl=fert_ci[fert_ci.tokenizer=='gpt4_cl100k']
best_enc=enc_ci.iloc[0]
rho_j_f=stats.spearmanr(df.joshi,df.fertility,nan_policy='omit')[0]
rho_j_r=stats.spearmanr(df.joshi,df.retrieval,nan_policy='omit')[0]
rho_axes=stats.spearmanr(df.fertility,df.retrieval,nan_policy='omit')[0]
worst_r=lang_ci.iloc[0]
txt=f'''# Results (auto-drafted from data)

## 5.1 Tokenizer fairness (Experiment 1)
On identical content across {df.shape[0]} languages, tokenizer fertility spread (max/min tokens per verse) ranges from {fert_ci.spread_maxmin.min():.1f}x to {fert_ci.spread_maxmin.max():.1f}x. Among generative-LLM tokenizers, GPT-4o (o200k) is fairest (spread {float(gpt4o.spread_maxmin.iloc[0]):.1f}x; mean {float(gpt4o.mean_tpv.iloc[0]):.1f} tok/verse, 95% CI [{float(gpt4o.mean_lo.iloc[0]):.1f}, {float(gpt4o.mean_hi.iloc[0]):.1f}]), while Llama-3 (spread {float(llama.spread_maxmin.iloc[0]):.1f}x) and GPT-4 cl100k (spread {float(cl.spread_maxmin.iloc[0]):.1f}x) remain the most unequal, carrying the full Brahmic-script penalty (e.g. Malayalam and Tamil exceed 250 tokens/verse under both). Llama-3 thus serves as a live negative example: a current, widely deployed model still imposing a 4-5x token tax on Brahmic-script users for the same content.

## 5.2 Retrieval benchmark (Experiment 2)
The verse alignment yields gold cross-lingual paraphrase pairs covering languages omitted by FLORES/Tatoeba. The strongest encoder, {best_enc.encoder}, reaches mean top-1 {best_enc.top1:.3f} (95% CI [{best_enc.top1_lo:.3f}, {best_enc.top1_hi:.3f}]). A nearest-neighbour probe shows embeddings organise by meaning, not language. Accuracy collapses on low-resource African languages -- the worst, {worst_r.language}, scores {worst_r.mean_top1:.3f} (95% CI [{worst_r.lo:.3f}, {worst_r.hi:.3f}]).

## 5.3 Two axes of inequity
Both axes correlate with resource level: Spearman(Joshi tier, fertility) = {rho_j_f:+.3f}; Spearman(Joshi tier, retrieval) = {rho_j_r:+.3f}. But the two axes are only moderately correlated with each other (Spearman = {rho_axes:+.3f}), and Kruskal-Wallis confirms script type drives fertility. Tokenizer cost and semantic coverage are therefore distinct dimensions of multilingual inequity: frontier tokenizers have largely closed the Brahmic tokenization gap, yet encoders still fail those and other low-resource languages. Fixing the tokenizer does not fix the encoder.
'''
open('results_section_draft.md','w').write(txt)
print(txt)

## 6. Bundle outputs

In [ ]:
out=['retrieval_encoder_ci.csv','retrieval_language_ci.csv','fertility_tokenizer_ci.csv',
     'joint_language_table.csv','table_fertility.tex','table_retrieval.tex','results_section_draft.md']
out=[o for o in out if os.path.exists(o)]
with zipfile.ZipFile('quran_stats_results.zip','w') as z:
    for o in out: z.write(o)
print('zipped:', out)
from google.colab import files; files.download('quran_stats_results.zip')

## 7. (Optional) Per-verse fertility bootstrap
For tighter, verse-level CIs on a chosen tokenizer, upload `translations.zip`, re-tokenize, and bootstrap over the 6,236 verses. Heavier; only needed if reviewers want per-verse CIs.

In [ ]:
# OPTIONAL — uncomment to run.
# !pip -q install tiktoken
# import tiktoken, csv
# from google.colab import files as _f
# print('upload translations.zip'); u=_f.upload()
# import io as _io
# os.makedirs('/content/tz',exist_ok=True)
# for fn in u:
#     if fn.endswith('.zip'): zipfile.ZipFile(_io.BytesIO(u[fn])).extractall('/content/tz')
# enc=tiktoken.get_encoding('o200k_base')
# def load(fp):
#     d={}; st=False
#     for row in csv.reader(open(fp,encoding='utf-8',errors='replace')):
#         if not st:
#             low=[(c or '').strip().lower() for c in row]
#             if 'sura' in low and 'aya' in low: st=True; iS,iA,iT=low.index('sura'),low.index('aya'),(low.index('translation') if 'translation' in low else 3)
#             continue
#         try: d[(int(row[iS]),int(row[iA]))]=row[iT]
#         except: pass
#     return d
# fp=glob.glob('/content/tz/**/english*.csv',recursive=True)[0]
# texts=list(load(fp).values())
# counts=np.array([len(t) for t in enc.encode_batch(texts)])
# bm=counts[rng.integers(0,len(counts),size=(5000,len(counts)))].mean(1)
# print('English GPT-4o tokens/verse %.2f  95%% CI [%.2f, %.2f]'%(counts.mean(),np.percentile(bm,2.5),np.percentile(bm,97.5)))